# Snow-Underlay video walkthrough

*Constants as the bridge — in motion.*

This notebook walks through the per-frame video pipeline that produces the canonical road-overlay clip. Two halves:

- **The story** (cells 1–7) — the principle, the architecture, the canonical artefact, with one figure each. Reads top-to-bottom as an essay.
- **Technical replication** (cells 8–end) — load a track, run the per-frame pipeline on a 5-second sub-clip, inspect intermediate state, render a small overlay. Re-executable end-to-end.

The full `make reproduce` end-to-end target (canonical 15-second clip from a clean clone, ~50 min on Mac CPU) lives at the project Makefile. This notebook is the *understandable* version, not the *master* one.

## 1. The principle

A snow plough's job is short: keep the road clear. While it's doing it, the road is invisible. A self-driving stack trained on Cityscapes will report, with calibrated confidence, that the entire scene is sky.

The familiar response is *we just need more data*. There are 27 million miles of road in the world; the long tail of conditions any of them can be in is longer than the road itself. We are not going to label our way out of it.

**The move**: for almost every operating regime where autonomy fails for lack of data, there is an adjacent regime — temporally, seasonally, geographically — where we have plenty of data, and where the parts that matter are the same. A snow plough's road is the same road it was last July. The road's *appearance* has changed completely; its *position in space* has not.

We bridge through what stays constant.

## 2. The recipe

Per snow frame:

1. Pull the live snowy frame.
2. Pull a clear-season prior of the same coordinates (Boreas paired summer drives for the canonical demo).
3. Match the two with a frozen feature matcher (DISK + LightGlue).
4. Fit a homography by RANSAC, biased toward the ground plane.
5. Segment the road on the *clear* prior only (Mask2Former, Cityscapes-pretrained).
6. Warp the road mask through the homography onto the snowy frame.

The plough now knows where the road is. No model in the pipeline has been trained on snow.

## 3. The video extension

The static-pair pipeline (`make stills`, the v1.x demo) works on individual snow / clear pairs. The video extension wraps it in three thin layers:

- **Track loader** — indexes a snow stream and a paired summer stream by GPS pose.
- **Prior pool** — picks the K = 3 nearest summer captures by UTM distance per snow frame; caches Mask2Former mask on first segmentation.
- **EMA temporal smoother** — α = 0.4 on the binary mask. Suppresses jitter without drifting.

Plus a pickled cache layer that makes matching (~50 min for a 15-second clip) reusable across renders that change only the smoother.

## 4. What we tried that didn't work

**Synthetic priors from past frames.** Snow→snow matching returns 3–7× more inliers than snow→summer because lighting / lens / viewpoint conditions are identical between consecutive snow frames. In stills, the resulting mask was dramatically broader. In motion, each frame's slightly-too-large mask seeded the next frame's prior, which produced a slightly-larger mask, and the road mask drifted outward into bushes over a few seconds. **A positive feedback loop the static stills hid.**

**Optical-flow propagation between matched keyframes.** Same outcome, different mechanism: vanishing-point flow stretches the previous mask outward each step.

**EMA on the binary mask** is what survived. Simplest possible smoother; does the least damage.

## 5. The canonical clip

Boreas `boreas-2021-01-26-11-22` — January 2021, Toronto suburbs, road buried, lane markings invisible. The cross-season pipeline produces a continuous green road overlay tracking the buried road frame by frame.

Watch: [`outputs/video/boreas_2021_01_26/overlay.mp4`](../outputs/video/boreas_2021_01_26/overlay.mp4) (15 s, ~7 MB).

---

## Technical replication

Below: load a track, run the per-frame pipeline on a 5-second sub-clip, inspect what it produced, render a small overlay. Self-contained — re-executes end-to-end after `make reproduce`.

**Prerequisites** (run from the project root, not the notebook):

```bash
uv sync --python 3.12
make reproduce  # one-time: pulls Boreas track + builds matching cache
```

The cells below load from the cache, so they execute in <60 s once the cache exists.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

# Project root for relative paths used by src.video_runtime imports.
PROJECT = Path.cwd()
while PROJECT != PROJECT.parent and not (PROJECT / 'pyproject.toml').exists():
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
PROJECT

In [ ]:
# Load the canonical Boreas track. The track abstracts a snow stream + a paired
# summer stream. `Track` reads the camera_poses.csv files written by
# data/video/tracks/<track_id>/{snow,summer}/, plus the on-disk frame PNGs.
from src.video_runtime.track import Track

track = Track('boreas_2021_01_26')
print(f'snow frames: {track.snow_frame_count()}')
print(f'summer frames: {len(track.summer_meta)}')
print(f'first snow pose: easting={track.snow_meta[0].easting:.1f}, '
      f'northing={track.snow_meta[0].northing:.1f}, '
      f'heading={track.snow_meta[0].heading:.2f}')

In [ ]:
# Show one snow frame + its UTM-nearest summer counterpart.
import matplotlib.pyplot as plt
from src.video_runtime.prior_pool import PriorPool

snow_meta = track.snow_meta[150]   # mid-window
snow_img = track.load_frame(snow_meta, max_dim=1024)

pool = PriorPool(track, K=1, max_dim=1024)
priors = pool.select(snow_meta)
summer = priors[0]
print(f'closest summer prior: {summer.distance_m:.2f} m from snow pose')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(snow_img); axes[0].set_title('snow query'); axes[0].axis('off')
axes[1].imshow(summer.image); axes[1].set_title(f'summer prior ({summer.distance_m:.2f} m away)')
axes[1].axis('off')
plt.tight_layout()

In [ ]:
# The cached Mask2Former road mask on the summer prior.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(summer.image); axes[0].set_title('summer prior'); axes[0].axis('off')
axes[1].imshow(summer.image); axes[1].imshow(summer.road_mask, alpha=0.5, cmap='Greens')
axes[1].set_title('summer + road mask (Cityscapes / Mask2Former)'); axes[1].axis('off')
plt.tight_layout()

In [ ]:
# Run the per-frame pipeline on a 5-second sub-clip (50 frames, stride 1).
# Loads from cache if it exists (~ 1 s); otherwise matches from scratch (~12 min).
from src.video_runtime.pipeline_v import run_track
from src.video_runtime.temporal import EMASmoother

results = run_track(
    'boreas_2021_01_26',
    K=3, max_dim=1024,
    start=100, end=150, stride=1,        # 50-frame slice
    foreground_y_frac=0.30,
    smoother=EMASmoother(alpha=0.4),
    cache_path=Path('outputs/video/boreas_2021_01_26/_cache_canonical.pkl'),
)
print(f'processed {len(results)} frames')
print(f'first frame: {results[0].n_priors_used} priors used, '
      f'inliers={[p["n_inliers"] for p in results[0].per_prior]}')

In [ ]:
# Inspect a few frames' overlays.
from src.overlay import alpha_blend
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, idx in zip(axes, [0, 15, 30, 45]):
    r = results[idx]
    canvas = r.snow_image.copy()
    if r.fused_mask is not None and int(r.fused_mask.sum()) > 0:
        canvas = alpha_blend(canvas, r.fused_mask, color=(0, 220, 100), alpha=0.45)
    ax.imshow(canvas); ax.set_title(f'frame {idx}'); ax.axis('off')
plt.tight_layout()

## What we didn't claim

We didn't train anything. We didn't fine-tune. We didn't collect a snow corpus. We didn't write a single line of snow-aware logic.

We didn't claim the system replaces lidar or 3D depth estimation. The output is a 2D road *prior*, not a drivable-surface estimate.

We didn't claim real-time. The matcher is the cost; the canonical 15-second clip's matching cache builds in ~50 min on Mac CPU. The cache makes that cost amortise across renders, but real-time would require a substantially faster correspondence model.

The novelty, such as it is, is in the *composition*: matcher, segmenter, homography are off-the-shelf; bridging through a constant is the move.

## Generalising

A model trained on regime A. An inference target in regime B. A known correspondence between the two. Transfer through the correspondence. Snow on a road is one instance. Polar imaging without polar training data, low-light medical imaging without low-light training data, a manipulator on Mars without Mars training data. Each admits the same structure.

We are not going to label our way out of every long-tail regime. For many of them, we don't have to.

*Constants as the bridge.*